# RAG Pipelines- Data ingestion to VectorDB pipeline

In [13]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_core import documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from sympy.physics.units import temperature

In [14]:
# Read all the Pdfs in the directory
def process_all_pdfs(pdf_directory):
    """" reads all pdf files in pdf_directory """
    all_docs= []
    pdf_dir = Path(pdf_directory)

    # finds all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            #load the PDF page by page
            loader = PyMuPDFLoader(pdf_file)
            pages = loader.load()

            #Tag each page so we know exactly which file it came from earlier
            for page in pages:
                page.metadata["source"] = pdf_file.name
            all_docs.extend(pages)

        except Exception as e:
            print(f"Error loading {pdf_file.name} : {e}")
    print(f"\nProcessed {len(all_docs)} PDF files")
    return all_docs

#process all PDFs ion the data directory
alldocs = process_all_pdfs("../data")






Found 3 PDF files to process
Processing ../data/pdf/doc_01_rag_and_langchain.pdf
Processing ../data/pdf/doc_02_vector_databases.pdf
Processing ../data/pdf/doc_03_prompting_and_guardrails.pdf

Processed 3 PDF files


In [15]:
print(alldocs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}, page_content="RAG + LangChain: Practical Notes\nRetrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in\nexternal documents. Instead of relying only on the model's parameters, a retriever selects\nrelevant chunks from a knowledge base and passes them to the LLM as context.\nIn LangChain, a typical pipeline has: (1) document loading, (2) text splitting, (3) embeddings, (4)\nvector store indexing, and (5) a retrieval + generation ch

In [16]:
### Test splitting into chunks

def document_split(documents, chunk_size = 1000, chunk_overlap=200):
    """ split documents into smaller chunks for better rag performance """

    # This object defines the rules for how we cut the text
    text_splitter = RecursiveCharacterTextSplitter(
        #the maximum number of character in one chunk
        chunk_size = chunk_size,
        #to keep context connected
        chunk_overlap = chunk_overlap,

        length_function = len,

        # priority list of where to cut
        separators = ["\n\n", "\n", " ", ""]
    )
    #This takes the list of PDF pages and returns a much longer list of chunks.
    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show preview of first chunk
    if split_docs:
        print(f"\n----Preview of first chunk 0 -----")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadta: {split_docs[0].metadata}")

    return split_docs

In [17]:
chunks = document_split(alldocs)
chunks

Split 3 documents into 4 chunks

----Preview of first chunk 0 -----
Content: RAG + LangChain: Practical Notes
Retrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in
external documents. Instead of relying only on the model's parameters, a retriever
Metadta: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}, page_content="RAG + LangChain: Practical Notes\nRetrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in\nexternal documents. Instead of relying only on the model's parameters, a retriever selects\nrelevant chunks from a knowledge base and passes them to the LLM as context.\nIn LangChain, a typical pipeline has: (1) document loading, (2) text splitting, (3) embeddings, (4)\nvector store indexing, and (5) a retrieval + generation ch

# Embedding and vectorStoreDB

In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import  cosine_similarity



In [19]:
class EmbeddingManager:
    """ handles document embeddings generation using SentenceTransformer """

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """ initialize the embedding manager """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """load sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            # This loads the neural network weights into the memory
            self.model = SentenceTransformer(self.model_name)

            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Converts english test chunks into list of numbers(vectors)
        Returns: A numPy array (matrix) where each row is different text chunk

        """
        #safety check to ensure model actually exists in ram
        if not self.model:
            raise ValueError("Nodel not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")

        # encode() is the math engine. It runs the text through the transformer layers.
        # show_progress_bar is perfect for tracking progress
        embeddings = self.model.encode(texts,show_progress_bar=True)

        # shape shows ( Number of chunks, Embedding dimension)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager





Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1769.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


# VectorStore

In [20]:
class VectorStore:
    """ Manages document embeddings in a chromaDB vector store """
    def __init__(self, collection_name:str="pdf_documents", persist_directory:str="../data/vector_store"):
        """
        Initialized the vector store
            :param collection_name: name of the ChromaDB collection
            :param persist_directory: Directory to persist the vecotr store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the chromaDB client and collection"""
        try:
            #create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error creating collection: {self.collection_name} : {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adds documents ad their embeddings to the collection
            :param documents: List of langchain documents
            :param embeddings: Corresponding embeddings
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to collection: {self.collection_name}")

        #Prepare data for chroma DB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ids
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare the metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            document_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                embeddings = embeddings_list,
                documents = document_text
            )
            print(f"Successfully added {len(ids)} documents to document vector store")
            print(f"Total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to collection: {self.collection_name}")
            raise

vectorstore = VectorStore()
vectorstore

Initialized collection: pdf_documents
Existing documents in collection: 16


In [21]:
### Convert the text to embeddings
texts= [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store it in the vector database
vectorstore.add_document(chunks,embeddings)

Generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to collection: pdf_documents
Successfully added 4 documents to document vector store
Total documents in collection : 20


# Retriever Pipeline from VectorStore


In [22]:
class RAGRetriever:
    """
    Its job is to take a human question, turn it into math, and find the most similar
    text chunks in the VectorStore.
    """
    def __init__(self, vector_store: 'VectorStore', embedding_manager:"EmbeddingManager"):
        """
        Connects the retriever to the store and manager
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int= 5, score_threshold:float =0.0) -> List[Dict[str,Any]]:
        """
        The main search function
        :param query: The question to search for
        :param top_k: How many best matches to return (default 5)
        :param score_threshold: A quality filter (0.0 to 1.0) 1.0 is perfect match
        :return List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embeddings =self.embedding_manager.generate_embeddings([query])[0]

        #Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings= [query_embeddings.tolist()],
                n_results=top_k
            )

            #Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance into similarity score (chromaDB uses cosine distance)
                    similarity_score = max(0, 1- distance)
                    print(f"Match {i+1}: Dist={distance:.4f}, Sim={similarity_score:.4f}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "doc_id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents for query: {query}: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)





In [23]:
print(vectorstore.collection.count())
rag_retriever.retrieve("rag and langchain")

20
Retrieving documents for query: rag and langchain
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.90it/s]

Generated embeddings with shape: (1, 384)
Match 1: Dist=1.4426, Sim=0.0000
Retrieved 1 documents (after filtering)
Match 2: Dist=1.4426, Sim=0.0000
Retrieved 2 documents (after filtering)
Match 3: Dist=1.4426, Sim=0.0000
Retrieved 3 documents (after filtering)
Match 4: Dist=1.4426, Sim=0.0000
Retrieved 4 documents (after filtering)
Match 5: Dist=1.4426, Sim=0.0000
Retrieved 5 documents (after filtering)


[{'doc_id': '50c65c0f_0',
  'document': "RAG + LangChain: Practical Notes\nRetrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in\nexternal documents. Instead of relying only on the model's parameters, a retriever selects\nrelevant chunks from a knowledge base and passes them to the LLM as context.\nIn LangChain, a typical pipeline has: (1) document loading, (2) text splitting, (3) embeddings, (4)\nvector store indexing, and (5) a retrieval + generation chain. The quality of chunking and\nmetadata often has more impact than model choice.\nChunking guidance: choose a chunk size that matches how your users ask questions. Use\noverlap to preserve context across boundaries. Store metadata such as source filename,\nsection headings, or timestamps so you can cite sources in responses.\nEvaluation tip: build a small set of question-answer pairs and check (a) retrieval recall and (b)\nfinal answer faithfulness. If retrieval is weak, tune embeddings, chunking, or t

# RAG Pipeline- VectorDB To LLM Output Generation

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
load_dotenv()

/home/riju/projects/rag-basics/rag-langchain-basics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [73]:
class GeminiLLM:
    # change the default model to Flash
    def __init__(self, model_name:str = "gemini-2.5-flash", api_key:str=None):

        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GEMINI_LLM_API_KEY")

        #Crash early if key is missing from .env file
        if not self.api_key:
            raise ValueError("Google API key is required.")

        # Initialize the gemini engine
        self.llm = ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model= self.model_name,
            temperature=0.1,  # Keep it at 0.1 so it does not hallucinate
            max_tokens=2024  # Limit response size to save API usage
        )

        print(f"Initialized Gemini LLM model: {self.model_name}")

    def generate_response(self, query:str, context:str) -> str:

        # Create the strict rulebook (Template stays exactly the same)
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""
            You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
            Context: {context}
            Question: {question}
            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say no
            """
        )

        #Inject the PDF chunks (context) and user's question into the brackets
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:

            # Wrap the formatted string as a 'Human' message
            messages = [HumanMessage(content= formatted_prompt)]

            # Send it to Google's servers and wait for the reply
            response = self.llm.invoke(messages)

            # Extract the raw english text from the response object
            return response.content

        except Exception as e:
            return f"Error generating response: {e}"



In [74]:
# Initialize Gemini LLM
try:
    gemini_llm = GeminiLLM(api_key=os.getenv("GEMINI_API_KEY"))
    print("Gemini LLM successfully Initialized")
except Exception as e:
    print(f"Warning:{e}. Gemini initialization failed")
    gemini_llm = None

Initialized Gemini LLM model: gemini-2.5-flash
Gemini LLM successfully Initialized


In [75]:
# Get the context from the retriever and pass it to the LLM

#Simple rag function: Retrieve context and generate response
def rag_simple(query, retriever , llm, top_k=3):

    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['document'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to  answer the question"

    #generate the answer using GEMINI LLM
    response = llm.generate_response(context, query)
    return response


In [76]:
answer = rag_simple("explain langchain in extreme detail" , rag_retriever, gemini_llm)
print(answer)

Retrieving documents for query: explain langchain in extreme detail
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.50it/s]

Generated embeddings with shape: (1, 384)
Match 1: Dist=1.4211, Sim=0.0000
Retrieved 1 documents (after filtering)
Match 2: Dist=1.4211, Sim=0.0000
Retrieved 2 documents (after filtering)
Match 3: Dist=1.4211, Sim=0.0000
Retrieved 3 documents (after filtering)


Retrieval-Augmented Generation (RAG) enhances Large Language Model (LLM) answers by grounding them in external documents. Instead of solely relying on the model's internal parameters, a retriever selects relevant chunks from a knowledge base and provides them to the LLM as context.

In LangChain, a typical RAG pipeline involves five steps:
1.  **Document loading:** Ingesting the source documents.
2.  **Text splitting:** Breaking down documents into smaller, manageable chunks.
3.  **Embeddings:** Converting these text chunks into numerical vector representations.
4.  **Vector store indexing:** Storing these embeddings in a vector database for efficient retrieval.
5.  **Retrieval + generation chain:** Retrieving relevant chunks based on a query and then using an LLM to generate an answer based on those chunks.

**Key considerations:**
*   The quality of **chunking and metadata** often has a greater impact on performance than the choice of the LLM itself.
*   **Chunking guidance:**
    * 